In [1]:
import os
import random
import html
from pathlib import Path
from IPython.display import display, HTML

In [2]:
native_dir = "./parallel_data/golden_speakers" # original LibriTTS data (native)
dorado_speakers_dir = "./data"

N_SAMPLES_PER_DATASET = 10
RANDOM_SAMPLE = True
RANDOM_SEED = 1
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

In [3]:
def get_speaker_dirs(data_root: Path):
    """
    Return {speaker_id: speaker_dir} from:
      <data_root>/<speaker_id>/
    """
    if not data_root.exists():
        return {}
    return {
        p.name: p
        for p in sorted(data_root.iterdir())
        if p.is_dir()
    }


def list_wavs(speaker_dir: Path):
    """
    Return audio files from:
      <speaker_dir>/wav/*.wav
    """
    wav_dir = speaker_dir / "wav"
    if not wav_dir.exists():
        return []

    return sorted([
        p for p in wav_dir.iterdir()
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS
    ])


def match_pairs(dorado_speaker_dir: Path, native_speaker_dir: Path):
    """
    Match by exact filename inside wav/
    """
    dorado_files = list_wavs(dorado_speaker_dir)
    native_files = list_wavs(native_speaker_dir)

    dorado_map = {p.name: p for p in dorado_files}
    native_map = {p.name: p for p in native_files}

    common_names = sorted(set(dorado_map) & set(native_map))
    return [(dorado_map[name], native_map[name], name) for name in common_names]


def choose_pairs(pairs, n_samples, random_sample=True, seed=42):
    if len(pairs) <= n_samples:
        return pairs
    if random_sample:
        rng = random.Random(seed)
        return rng.sample(pairs, n_samples)
    return pairs[:n_samples]


def audio_player_html(audio_path, width="260px"):
    p = Path(audio_path)
    if not p.exists():
        return f"""
        <div style="color:#a33; font-size:13px; padding:8px; text-align:center;">
            Missing:<br>{html.escape(str(audio_path))}
        </div>
        """
    return f"""
    <audio controls preload="none" style="width:{width};">
        <source src="{html.escape(p.as_posix())}">
        Your browser does not support the audio element.
    </audio>
    """

def build_dataset_table_html(dataset_name, pairs):
    title_html = f"""
    <div style="
        font-family: Arial, sans-serif;
        font-size: 24px;
        font-weight: 700;
        margin: 26px 0 10px 0;
        color: #white;
    ">
        {html.escape(dataset_name)}
    </div>
    """

    table_html = """
    <table class="audio-table">
        <tr>
            <th style="width:260px;">File</th>
            <th>Native speaker (LibriTTS)</th>
            <th>Dorado speaker</th>
        </tr>
    """

    for native_path, dorado_path, filename in pairs:
        table_html += f"""
        <tr>
            <td class="audio-row-label">{html.escape(filename)}</td>
            <td>{audio_player_html(native_path)}</td>
            <td>{audio_player_html(dorado_path)}</td>
        </tr>
        """

    table_html += "</table>"
    return title_html + table_html

# =========================================================
# 3) CSS
# =========================================================
GLOBAL_CSS = """
<style>
.audio-table {
    border-collapse: collapse;
    width: 100%;
    max-width: 1100px;
    font-family: Arial, sans-serif;
    table-layout: fixed;
    margin-bottom: 24px;
}
.audio-table th, .audio-table td {
    border: 2px solid white;
    text-align: center;
    vertical-align: middle;
    padding: 14px 10px;
    word-wrap: break-word;
}
.audio-table th {
    background: #b63b6d;
    color: white;
    font-size: 18px;
    font-weight: 700;
}
.audio-table td {
    background: #dccfd4;
    color: #222;
}
.audio-table tr:nth-child(even) td {
    background: #e8dfe2;
}
.audio-row-label {
    background: #f3f0f1 !important;
    font-weight: 700;
    font-size: 13px;
}
.audio-table audio {
    width: 85%;
    min-width: 180px;
}
</style>
"""
display(HTML(GLOBAL_CSS))

In [4]:
all_html = ""
dorado_speakers_root = Path(dorado_speakers_dir)
native_root = Path(native_dir)

dorado_speakers = get_speaker_dirs(dorado_speakers_root)
native_speakers = get_speaker_dirs(native_root)

common_speakers = sorted(set(dorado_speakers) & set(native_speakers))

dataset_pairs = []
for speaker_id in common_speakers:
    dataset_pairs.extend(
        match_pairs(
            native_speakers[speaker_id],
            dorado_speakers[speaker_id]
        )
    )

chosen_pairs = choose_pairs(
    dataset_pairs,
    n_samples=N_SAMPLES_PER_DATASET,
    random_sample=RANDOM_SAMPLE,
    seed=RANDOM_SEED,
)

if not chosen_pairs:
    all_html += f"""
    <div style="font-family: Arial, sans-serif; margin: 24px 0;">
        <div style="font-size:24px; font-weight:700;  color: white;">SyntheticSpanish_v1</div>
        <div style="color:#a33; margin-top:8px;">
            No matched native / dorado speaker pairs found.
        </div>
    </div>
    """
else:
    all_html += build_dataset_table_html("SyntheticSpanish_v1", chosen_pairs)

display(HTML(all_html))

File,Native speaker (LibriTTS),Dorado speaker
2407_7666_000002_000001.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
667_107247_000036_000000.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
8590_258292_000011_000000.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
8208_256239_000024_000000.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
1665_136991_000039_000003.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
3660_6517_000056_000000.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
2230_148550_000019_000000.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
614_6500_000019_000001.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
8208_256238_000011_000004.wav,Your browser does not support the audio element.,Your browser does not support the audio element.
5893_54390_000060_000000.wav,Your browser does not support the audio element.,Your browser does not support the audio element.


In [5]:
import json
json_data = {
    'samples': []
}
out_folder = Path("demo")
out_folder.mkdir(exist_ok=True)
for i, (native_path, dorado_path, filename) in enumerate(chosen_pairs):
    basename = "sample_{:02d}".format(i+1)
    out_sample_dir = out_folder / basename
    out_sample_dir.mkdir(exist_ok=True)
    out_native_path = out_sample_dir / "native.wav"
    out_dorado_path = out_sample_dir / "dorado.wav"
    os.system(f"cp {native_path} {out_native_path}")
    os.system(f"cp {dorado_path} {out_dorado_path}")
    json_data['samples'].append({
        "id": basename,
        "label": " ".join(basename.capitalize().split("_"))
    })
with open(out_folder / "manifest.json", "w") as f:
    json.dump(json_data, f, indent=4)